In [0]:
%sql
-- ILOSTAT · Paso 1 (estructura): filas de cada tabla y columnas de ambas (comparar con S4).
-- Esperado: luu 18.972 filas y 276 ref_area; neet 13.829 filas y 276 ref_area (D16).
SELECT 'ilostat_luu' AS tabla, COUNT(*) AS filas FROM happiness.bronze.ilostat_luu
UNION ALL
SELECT 'ilostat_neet', COUNT(*) FROM happiness.bronze.ilostat_neet;

In [0]:
%sql
-- ILOSTAT · Paso 1 (estructura): columnas y tipos de las dos tablas (todo STRING salvo _ingest_ts).
-- neet debe traer note_indicator y note_source de más (D16).
SELECT table_name, ordinal_position, column_name, data_type
FROM happiness.information_schema.columns
WHERE table_schema = 'bronze' AND table_name IN ('ilostat_luu', 'ilostat_neet')
ORDER BY table_name, ordinal_position;

In [0]:
%sql
-- ILOSTAT · Paso 1 (estructura): muestra de cada tabla.
SELECT * FROM happiness.bronze.ilostat_luu LIMIT 5;

SELECT * FROM happiness.bronze.ilostat_neet LIMIT 5;

In [0]:
%sql
-- ILOSTAT · Paso 2 (nulos): claves y valor en las dos tablas. NULL real, vacío y texto 'null' por separado.
SELECT 'ilostat_luu' AS tabla, COUNT(*) AS filas,
       COUNT_IF(ref_area IS NULL OR ref_area IN ('', 'null'))     AS ref_area_faltante,
       COUNT_IF(source IS NULL OR source IN ('', 'null'))         AS source_faltante,
       COUNT_IF(sex IS NULL OR sex IN ('', 'null'))               AS sex_faltante,
       COUNT_IF(`time` IS NULL OR `time` IN ('', 'null'))         AS anio_faltante,
       COUNT_IF(obs_value IS NULL)                                AS valor_null_real,
       COUNT_IF(obs_value IN ('', 'null'))                        AS valor_vacio_o_texto,
       COUNT_IF(obs_status IS NULL)                               AS status_null_real,
       COUNT_IF(_rescued_data IS NOT NULL)                        AS filas_rescatadas
FROM happiness.bronze.ilostat_luu
UNION ALL
SELECT 'ilostat_neet', COUNT(*),
       COUNT_IF(ref_area IS NULL OR ref_area IN ('', 'null')),
       COUNT_IF(source IS NULL OR source IN ('', 'null')),
       COUNT_IF(sex IS NULL OR sex IN ('', 'null')),
       COUNT_IF(`time` IS NULL OR `time` IN ('', 'null')),
       COUNT_IF(obs_value IS NULL),
       COUNT_IF(obs_value IN ('', 'null')),
       COUNT_IF(obs_status IS NULL),
       COUNT_IF(_rescued_data IS NOT NULL)
FROM happiness.bronze.ilostat_neet;

In [0]:
%sql
-- ILOSTAT · Paso 3 (cardinalidad): valores distintos de sexo y de estado de la observación, por tabla.
-- 'sex' dice qué filas hay que quedarse en Silver (el total, no hombres/mujeres por separado).
-- 'obs_status' son las marcas de ILOSTAT (por ejemplo estimación o quiebre de serie).
SELECT 'ilostat_luu' AS tabla, sex, obs_status, COUNT(*) AS filas FROM happiness.bronze.ilostat_luu GROUP BY ALL
UNION ALL
SELECT 'ilostat_neet', sex, obs_status, COUNT(*) FROM happiness.bronze.ilostat_neet GROUP BY ALL
ORDER BY tabla, sex, obs_status;

In [0]:
%sql
-- ILOSTAT · Paso 3 (cardinalidad): fuentes de datos, indicadores y países distintos, y rango de años por tabla.
-- Hay una columna 'source' (la encuesta o registro de donde sale el dato): un mismo país-año puede venir de varias.
SELECT 'ilostat_luu' AS tabla, COUNT(DISTINCT source) AS fuentes, COUNT(DISTINCT indicator) AS indicadores,
       COUNT(DISTINCT ref_area) AS paises, MIN(TRY_CAST(`time` AS INT)) AS primer_anio, MAX(TRY_CAST(`time` AS INT)) AS ultimo_anio
FROM happiness.bronze.ilostat_luu
UNION ALL
SELECT 'ilostat_neet', COUNT(DISTINCT source), COUNT(DISTINCT indicator),
       COUNT(DISTINCT ref_area), MIN(TRY_CAST(`time` AS INT)), MAX(TRY_CAST(`time` AS INT))
FROM happiness.bronze.ilostat_neet;

In [0]:
%sql
-- ILOSTAT · Paso 5 (duplicados), dos niveles:
--  (a) clave completa (ref_area, source, indicator, sex, time): esperado 0 repetidas.
--  (b) país-año-sexo con más de una fuente: NO es error, pero Silver tendrá que elegir una fuente por país-año.
WITH luu AS (
  SELECT ref_area, source, indicator, sex, `time`,
         ROW_NUMBER() OVER (PARTITION BY ref_area, source, indicator, sex, `time` ORDER BY _ingest_ts) AS rn
  FROM happiness.bronze.ilostat_luu
), luu_fuentes AS (
  SELECT ref_area, sex, `time`, COUNT(DISTINCT source) AS n_fuentes FROM happiness.bronze.ilostat_luu GROUP BY ALL
), neet AS (
  SELECT ref_area, source, indicator, sex, `time`,
         ROW_NUMBER() OVER (PARTITION BY ref_area, source, indicator, sex, `time` ORDER BY _ingest_ts) AS rn
  FROM happiness.bronze.ilostat_neet
), neet_fuentes AS (
  SELECT ref_area, sex, `time`, COUNT(DISTINCT source) AS n_fuentes FROM happiness.bronze.ilostat_neet GROUP BY ALL
)
SELECT 'ilostat_luu' AS tabla,
       (SELECT COUNT_IF(rn > 1) FROM luu)              AS claves_duplicadas,
       (SELECT COUNT_IF(n_fuentes > 1) FROM luu_fuentes) AS pais_anio_sexo_con_varias_fuentes,
       (SELECT COUNT(*) FROM luu_fuentes)              AS pais_anio_sexo_total
UNION ALL
SELECT 'ilostat_neet',
       (SELECT COUNT_IF(rn > 1) FROM neet),
       (SELECT COUNT_IF(n_fuentes > 1) FROM neet_fuentes),
       (SELECT COUNT(*) FROM neet_fuentes);

In [0]:
%sql
-- ILOSTAT · Paso 4 (rangos): valor observado (son porcentajes, esperado entre 0 y 100) y no numéricos.
SELECT 'ilostat_luu' AS tabla, MIN(TRY_CAST(obs_value AS DOUBLE)) AS minimo, MAX(TRY_CAST(obs_value AS DOUBLE)) AS maximo,
       COUNT_IF(obs_value NOT IN ('', 'null') AND TRY_CAST(obs_value AS DOUBLE) IS NULL) AS no_numericos
FROM happiness.bronze.ilostat_luu
UNION ALL
SELECT 'ilostat_neet', MIN(TRY_CAST(obs_value AS DOUBLE)), MAX(TRY_CAST(obs_value AS DOUBLE)),
       COUNT_IF(obs_value NOT IN ('', 'null') AND TRY_CAST(obs_value AS DOUBLE) IS NULL)
FROM happiness.bronze.ilostat_neet;

In [0]:
%sql
-- ILOSTAT · Paso 6 (temporal): países distintos con dato por año, en cada tabla.
-- Sirve para ver desde qué año la serie es usable y si el final del panel (2024-2025) está cubierto.
SELECT 'ilostat_luu' AS tabla, TRY_CAST(`time` AS INT) AS anio, COUNT(DISTINCT ref_area) AS paises
FROM happiness.bronze.ilostat_luu GROUP BY ALL
UNION ALL
SELECT 'ilostat_neet', TRY_CAST(`time` AS INT), COUNT(DISTINCT ref_area)
FROM happiness.bronze.ilostat_neet GROUP BY ALL
ORDER BY tabla, anio;

In [0]:
%sql
-- ILOSTAT · Para dim_pais_base: códigos ref_area que no existen en Banco Mundial (agregados regionales y similares).
-- Se junta lo de las dos tablas y se muestra cuántas filas trae cada código.
WITH ilo AS (
  SELECT ref_area FROM happiness.bronze.ilostat_luu
  UNION ALL
  SELECT ref_area FROM happiness.bronze.ilostat_neet
),
bm AS (SELECT DISTINCT `Country Code` AS codigo FROM happiness.bronze.banco_mundial)
SELECT i.ref_area, COUNT(*) AS filas
FROM ilo i LEFT JOIN bm ON i.ref_area = bm.codigo
WHERE bm.codigo IS NULL
GROUP BY i.ref_area
ORDER BY i.ref_area;

## Conclusiones ILOSTAT
- luu 18.972 filas, neet 13.829 filas; 276 ref_area en cada una. 0 duplicados; cada país-año-sexo viene de una sola fuente.
- Sexo: SEX_T/M/F (+ SEX_O en 9 filas de NEET). Silver filtra SEX_T.
- obs_status NULL real o marca: R (4.118 filas de luu, por confirmar), B (632 en neet), U (29 en neet). 14 valores nulos en neet.
- Rangos en %: luu 0,88-49,91; neet 2,02-86,79. 0 no numéricos.
- luu: 2005-2027, casi completo. neet: 1970-2027, esparso; 2026-2027 solo trae los 87 agregados.
- 98 ref_area fuera de Banco Mundial: 87 agregados (X..), CHA→CHI, KOS→XKX, TWN, ANT (retirado) y 6 territorios.
- obs_status: NULL real o marca. R (4.118 filas de luu) aparece en 2005-2024 (13%-35% de los países por año) y no en 2025-2027; significado por confirmar. B (632 en neet) = quiebre de serie; U (29 en neet).
- Años 2026-2027 son futuros y 2025 repite los mismos 271 países: estimaciones modeladas, no observaciones. Silver no usa 2026-2027; 2025 a decidir.

In [0]:
%sql
-- ILOSTAT · ¿Las marcas 'R' de luu son las proyecciones de los años recientes?
-- Se mira solo el total (SEX_T): por año, cuántas filas tienen marca R y cuántas no.
SELECT `time` AS anio, obs_status, COUNT(*) AS filas
FROM happiness.bronze.ilostat_luu
WHERE sex = 'SEX_T'
GROUP BY ALL
ORDER BY anio, obs_status;